# Build 04-02 · SHAP-DiD — cross-version estimate, within-version confound check, v2↔v3 shared claims

**Rebuilt 2026-09-13** after a long design discussion (see `project_shap_did_session2_updates`
and the ones after it in memory). Four sections, each independent enough to run on its own once
its prerequisites exist:

1. **Dose context** — how much of each version's training label was forced by the prior version.
2. **`cross_version_estimate`** — region(A/B) x version(before/after), on each version's own OOT
   split. Runs for v1->v2 and v2->v3. Same mechanism both times; role (does a non-zero DiD argue
   against parallel trends, or is it a real estimate) is read from the printed dose numbers, not
   auto-classified — see the section markdown for why. Followed by `local_cross_version_estimate`
   (2026-09-13 addition), the same mechanism restricted to a grid-selected band around τ — a
   sharp Regression Discontinuity Design (RDD) whose `simpson(B') - simpson(A')` at each version's
   own τ is a boundary discontinuity in the group-level concentration functional, used here as a
   robustness check on whether region A/B's differing covariate distributions, not the
   forced-label mechanism, are what moves the headline number.
3. **`within_version_confound_check`** — v3's TRAIN split only, split early/late. A fast check for
   whether SOMETHING (not necessarily forced-label dose) makes region A and B drift apart even
   within one version's own window. Points at `04_04_regime_break_robustness.ipynb` for the deep
   dive once this flags something.
4. **v2<->v3 shared claims** — score drift + a combined dose table, using `corrector_targets`
   (already joins v3's claims to v2's recorded score+decision by claim_id — no new join needed).
   A stretch-goal placeholder for a FULL shared-claims SHAP comparison closes the notebook.

In [ ]:
# §0 — setup (analysis .venv kernel)
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import schema
import figstyle
import feature_alias
from loaders import load
from estimator import concentration

figstyle.apply()
pd.set_option("display.width", 160)
print("ROOT =", ROOT)

## Dose context — how much of each version's training label was forced by the PRIOR version

`dose(v)` = the share of v's OWN training claims whose label came from the immediately prior
version's scrap decision (never garage-verified) — literally the PRIOR model's own scrap rate,
measured on the population that became v's training set (every scrapped claim is by definition
unverified, so "scrap rate" and "forced-label share" are the same number here). NOT v's own
current scrap rate on its own data — that answers a different question. Printed before any DiD
number below, because both sections after this need it to be interpretable.

In [ ]:
# dose(v) via corrector_targets. dose(v1) = 0 by construction: no prior model exists to have
# forced a v1 label (corrector_targets is not built for v1 at all).
def dose(version: str, split: str) -> float | None:
    p = config.split_path("corrector_targets", version, split)
    if not p.exists():
        print(f"[{version}] no corrector_targets at {p} -- dose unmeasured, not zero")
        return None
    ct = pd.read_parquet(p)
    return float((ct[schema.DECISION] == 1).mean())


dose_rows = []
for v in ("v2", "v3"):
    for split in list(dict.fromkeys(["train", config.OOT_SPLIT[v]])):
        d = dose(v, split)
        dose_rows.append({"version": v, "split": split, "dose": d})
        print(f"dose({v}, {split}) = {d:.4%}" if d is not None else f"dose({v}, {split}) = unmeasured")
print("\ndose(v1) = 0 by construction -- no PRIOR model exists to have forced any v1 label "
      "(pre_ml_label is the human-handler era); corrector_targets is not built for v1.")

dose_table = pd.DataFrame(dose_rows).set_index(["version", "split"])
figstyle.save_table(dose_table, "04_02_dose_context")
display(dose_table)

## `cross_version_estimate` — region(A/B) x version(before/after), on each version's own OOT split

**Design**: NO era axis. 2x2 = region(A/B, that version's own decision rule) x which version
(before/after). Runs on each version's own **out-of-time holdout** (`config.OOT_SPLIT`: v1
`val2`, v2 `test`, v3 `oot`) — never `train` — because SHAP on training data can reflect
overfitting/regularisation artefacts (v2 `reg_alpha=20` vs v3 `reg_alpha=0`,
`project_v2_xgb_config`) that have nothing to do with forced-label dose, and because comparing an
in-sample split against a holdout is itself a confound `concentration.require_comparable()` warns
about.

**On "placebo" vs "estimate" — one mechanism, not two names.** `cross_version_estimate()` computes
the exact same thing for v1->v2 and v2->v3: `DiD = (B-A)|after - (B-A)|before`. Whether a given
call's number should be read as a **validity check** (a placebo test: apply the estimator where
dose barely changes, see if it still returns something big — if so, something OTHER than dose is
moving region B against region A, e.g. the v1->v2 era/label-source jump) or as a **substantive
estimate** (v2->v3, where dose genuinely differs 20x by the synthetic pattern) depends entirely on
the two printed dose numbers, judged by hand once real doses are in — not on a separate function
name or an auto-threshold. An earlier version of this notebook tried a threshold + ratio rule and
it broke (v1's dose is hard-coded to exactly 0.0, so a ratio-based rule divides by zero and always
calls v1->v2 "not a placebo" -- backwards). Picking a threshold that works needs knowing the real
dose SCALE first, which is not measured yet — so this section prints the numbers and stops there.

In [ ]:
# §1 — helpers. ID_COL / TAG_COLS are shared with the confound-check section below.
ID_COL = schema.CLAIM_ID
TAG_COLS = [schema.DATE, "score", schema.DECISION, "region", "era", schema.OBSERVED]


def region_tag_inline(version: str, split: str) -> pd.DataFrame:
    """region(A/B) built in memory, for versions with no shap_did_input file (v1 only -- it is
    deliberately excluded from 04_01's BUILD, see 04_01 §1)."""
    d = load(version, split=split)
    df = d.frame.copy()
    df[schema.DECISION] = d.decisions
    df["region"] = np.where(df[schema.DECISION] == 1, "B", "A")
    return df[[ID_COL, "region"]]


def version_pair_table(version: str, split: str) -> pd.DataFrame:
    """region-tagged claims joined to that (version, split)'s attributions -- via 04_01's
    shap_did_input file if it exists (v2, v3), else built inline (v1)."""
    if config.split_path("shap_did_input", version, split).is_file():
        tags = pd.read_parquet(config.split_path("shap_did_input", version, split))
    else:
        tags = region_tag_inline(version, split)
    attrs = load(version, split=split).attributions
    return tags.merge(attrs, on=ID_COL, how="inner")


def cell_simpson(table: pd.DataFrame, **filters) -> float:
    """Simpson index of mean|phi|, restricted to the subset matching **filters."""
    sub = table
    for col, val in filters.items():
        sub = sub[sub[col] == val]
    drop_cols = [c for c in TAG_COLS if c in sub.columns]
    mabs = concentration.mean_abs(sub.drop(columns=drop_cols, errors="ignore"), id_col=ID_COL)
    return concentration.simpson(mabs)


def cell_mabs(table: pd.DataFrame, **filters) -> pd.Series:
    """mean|phi| per feature for the subset matching **filters -- the per-feature twin of
    cell_simpson(), same filtering rule."""
    sub = table
    for col, val in filters.items():
        sub = sub[sub[col] == val]
    drop_cols = [c for c in TAG_COLS if c in sub.columns]
    return concentration.mean_abs(sub.drop(columns=drop_cols, errors="ignore"), id_col=ID_COL)


def cross_version_estimate(v_before: str, split_before: str, v_after: str, split_after: str) -> dict:
    """region(A/B) x version(before/after) DiD, each version on its OWN OOT split."""
    table_before = version_pair_table(v_before, split_before)
    table_after = version_pair_table(v_after, split_after)

    try:
        concentration.require_comparable({
            v_before: load(v_before, split=split_before).attribution_meta,
            v_after: load(v_after, split=split_after).attribution_meta,
        })
    except ValueError as exc:
        print(f"NOTE: {v_before}/{v_after} attribution settings differ -- an EXTRA caveat, "
              f"not fixed here:")
        print(f"  {exc}\n")

    sA_before, sB_before = cell_simpson(table_before, region="A"), cell_simpson(table_before, region="B")
    sA_after, sB_after = cell_simpson(table_after, region="A"), cell_simpson(table_after, region="B")
    did = (sB_after - sA_after) - (sB_before - sA_before)

    dose_before = 0.0 if v_before == "v1" else dose(v_before, split_before)
    dose_after = dose(v_after, split_after)
    fmt = lambda x: f"{x:.4%}" if x is not None else "unmeasured"

    print(f"{v_before}/{split_before}: simpson(A)={sA_before:.4f}  simpson(B)={sB_before:.4f}  "
          f"(B-A)={sB_before - sA_before:+.4f}  n={len(table_before):,}  dose={fmt(dose_before)}")
    print(f"{v_after}/{split_after}: simpson(A)={sA_after:.4f}  simpson(B)={sB_after:.4f}  "
          f"(B-A)={sB_after - sA_after:+.4f}  n={len(table_after):,}  dose={fmt(dose_after)}")
    print(f"\nDiD({v_before}\u2192{v_after}) = {did:+.4f}")
    if dose_before is not None and dose_after is not None:
        print(f"dose_before={dose_before:.4%}  dose_after={dose_after:.4%}  "
              f"abs diff={abs(dose_after - dose_before):.4%}")
    print("-> read dose_before/dose_after against DiD by hand: both negligible for this business "
          "AND close to each other -> a non-zero DiD argues against parallel trends. A real, "
          "substantially different dose on at least one side -> read the DiD as an estimate.")

    return {"pair": f"{v_before}({split_before})->{v_after}({split_after})", "did": did,
            "dose_before": dose_before, "dose_after": dose_after,
            "n_before": len(table_before), "n_after": len(table_after)}

In [ ]:
# §2 — run it: v1->v2 (context: is this design's DiD ~0 when dose barely changes?) and v2->v3
# (the actual estimate), each version on its OWN out-of-time holdout
OOT = config.OOT_SPLIT   # {"v1": "val2", "v2": "test", "v3": "oot"}

estimate_v1_v2 = cross_version_estimate("v1", OOT["v1"], "v2", OOT["v2"])
print()
estimate_v2_v3 = cross_version_estimate("v2", OOT["v2"], "v3", OOT["v3"])

cross_version_table = pd.DataFrame([estimate_v1_v2, estimate_v2_v3]).set_index("pair")
figstyle.save_table(cross_version_table, "04_02_cross_version_estimate")
display(cross_version_table)

## `local_cross_version_estimate` — an RDD boundary discontinuity at τ, DiD'd across versions

`cross_version_estimate` above compares the WHOLE garage-verified population (region A) against
the WHOLE model-scrapped population (region B). The thesis already names the weak point of that
design (`subsec:shap-did`): A and B do not share a covariate distribution
(`p(x|s<=tau)` vs `p(x|s>tau)`), so a non-parallel trend in that distribution alone — with no
forced-label mechanism at all — could move the DiD. `notebook/real/00_SHAP.ipynb` §9b/§9c already
runs a WITHIN-version version of this idea (compare the `N_NEAR_ABOVE`/`N_NEAR_BELOW` claims
closest to τ on each side), and the thesis already names that construction precisely
(`Delta phi_boundary`, Eq. `eq:boundary-gap`, `sec:bg-causal`): "a SHAP-space analogue of
`tau_SRD` itself, differencing attribution concentration just above and just below τ **in place
of an outcome**" — i.e. a sharp Regression Discontinuity Design (RDD), with concentration
standing in for the outcome. This section is the missing CROSS-version twin of that same
construction: the same `(B-A)|after - (B-A)|before` mechanism, but with region redefined as a
narrow band `|score - tau| <= h` around each row's OWN τ, so the two sides being compared are
claims that scored almost identically and differ mainly in which side of the cutoff they fell on.
`simpson(B') - simpson(A')` at ONE version's own τ is that version's own boundary discontinuity,
exactly as Eq. `eq:boundary-gap` is; DiDing two such discontinuities across v2->v3 is what this
function computes.

**Precise about what this IS and ISN'T.** This is analogous to a Local Average Treatment Effect
(LATE), not a literal one. A textbook LATE averages a PER-UNIT treatment effect (unit-level
potential outcomes). `simpson(B') - simpson(A')` is built from each side's POOLED mean-`|phi|`
profile (Eq. `eq:meanabsshap` — the thesis's own headline attribution summary, already defined at
the group level, used identically in `cross_version_estimate`'s whole-population numbers above),
never from an average of each claim's OWN concentration. Simpson is convex, so "Simpson of the
group's average profile" and "average of each claim's own Simpson" are genuinely different
quantities that can even move in opposite directions (toy case: two claims, one 100% on feature
X, one 100% on feature Y — individually each is maximally concentrated, but pooled first, the
group's mean profile splits 50/50 and reads as LESS concentrated). What this section answers is
"does this region's POPULATION converge onto a shared narrow feature set", which is exactly the
mechanism the SFP hypothesis describes (the next model learns the SAME few features that drove
the fast-track decision) — not "is a typical individual claim's own reasoning narrow". Kept as
the group-level construction deliberately, so it stays comparable to `cross_version_estimate`'s
whole-population number above, which uses the identical (pooled-then-Simpson) definition.
`subsec:shap-did`'s "narrow to an overlapping band around τ ... [an RDD-based] local DiD"
mitigation is what this section builds — not yet built until now.

**h is selected, not assumed.** `band_h`'s published default (0.01, `mitigator/corrector/reweight.py`)
is only `03_02_reweight_mitigation.ipynb` §2c's FALLBACK constant for when its own selection
procedure fails — it has no independent justification for τ-boundary width in general, so it is
not reused as a number here. What IS reused is the SAME oracle-free, power-gated PROCEDURE
(`project_reweight_corrector_0302`): grid h upward, pick the smallest value at which every
`region_local x version` cell holds at least `LOCAL_MIN_CELL_N` claims, capped at `LOCAL_H_MAX` so
the gate cannot be satisfied by de-localising "local" into most of the population. Re-derived on
THIS notebook's own OOT populations, exactly as 03_02 re-derives its own `band_h` per training
split rather than sharing one number across contexts.

**v1 is excluded here too.** Its scrap rule is mobility-segmented (`threshold.apply`, not a scalar
τ), and the mobility field needed to resolve it is permanently unrecoverable
(`project_v1_mobility_not_a_feature`) — there is no single number to measure `|score - tau|`
against, so only v2→v3 gets a local estimate.

In [ ]:
# §2b — local RDD boundary discontinuity at tau, DiD'd across versions: an overlapping band
# around tau, v2->v3 only (see markdown -- v1's segmented rule has no scalar tau to band around)
LOCAL_H_GRID = [0.005, 0.0075, 0.01, 0.015, 0.02, 0.03, 0.05, 0.075, 0.1]
LOCAL_MIN_CELL_N = 100    # same power-gate rationale as 03_02 SS2c: RSE ~ 1/sqrt(n), 10% at n=100
LOCAL_H_MAX = 0.05        # same locality cap -- otherwise the gate is met by de-localising "local"
LOCAL_H_FALLBACK = 0.01   # 03_02's own fallback constant, reused only as a last resort here too


def _tau_by_date(table: pd.DataFrame, version: str) -> pd.Series:
    """tau looked up per UNIQUE date, not per row -- a version's OOT split alone can run to the
    order of 10^5 claims, and calling config.threshold_on row-by-row would be the slow way to ask
    a handful of regimes the same question that many times."""
    dates = pd.to_datetime(table[schema.DATE])
    tau_by_date = {d: config.threshold_on(version, d) for d in dates.unique()}
    return dates.map(tau_by_date)


def local_region(table: pd.DataFrame, version: str, h: float) -> pd.DataFrame:
    """Re-tags rows into region_local A'/B' by distance to THAT ROW's OWN tau (regime-aware for
    v2, single global for v3), keeping only |score - tau| <= h. Strict '>' matches the project's
    region convention (src/threshold.py): exactly at the band edge above tau is B', not A'."""
    dist = table["score"] - _tau_by_date(table, version)
    band = table[dist.abs() <= h].copy()
    band["region_local"] = np.where(dist[dist.abs() <= h] > 0, "B", "A")
    return band


def cell_simpson_local(table: pd.DataFrame, **filters) -> float:
    """cell_simpson's twin for region_local tables -- also drops region_local itself before
    computing mean|phi|, since it is not in TAG_COLS and would otherwise be read as a feature."""
    sub = table
    for col, val in filters.items():
        sub = sub[sub[col] == val]
    drop_cols = [c for c in TAG_COLS + ["region_local"] if c in sub.columns]
    mabs = concentration.mean_abs(sub.drop(columns=drop_cols, errors="ignore"), id_col=ID_COL)
    return concentration.simpson(mabs)


def cell_mabs_local(table: pd.DataFrame, **filters) -> pd.Series:
    sub = table
    for col, val in filters.items():
        sub = sub[sub[col] == val]
    drop_cols = [c for c in TAG_COLS + ["region_local"] if c in sub.columns]
    return concentration.mean_abs(sub.drop(columns=drop_cols, errors="ignore"), id_col=ID_COL)


def select_local_h(v_before: str, split_before: str, v_after: str, split_after: str,
                    grid=LOCAL_H_GRID, min_n=LOCAL_MIN_CELL_N, h_max=LOCAL_H_MAX,
                    fallback=LOCAL_H_FALLBACK) -> tuple[float, pd.DataFrame]:
    """Grid search on CELL COUNTS ONLY -- never on a concentration or precision number computed
    from these rows (the same discipline 03_02 SS2c uses for band_h/clip_hi)."""
    table_before = version_pair_table(v_before, split_before)
    table_after = version_pair_table(v_after, split_after)
    dist_before = table_before["score"] - _tau_by_date(table_before, v_before)
    dist_after = table_after["score"] - _tau_by_date(table_after, v_after)

    rows = []
    for h in sorted(grid):
        in_before, in_after = dist_before.abs() <= h, dist_after.abs() <= h
        n = {"A_before": int((dist_before[in_before] <= 0).sum()),
             "B_before": int((dist_before[in_before] > 0).sum()),
             "A_after": int((dist_after[in_after] <= 0).sum()),
             "B_after": int((dist_after[in_after] > 0).sum())}
        rows.append({"h": h, **n, "all_pass": min(n.values()) >= min_n})
    bt = pd.DataFrame(rows).set_index("h")

    ok = bt.index[bt["all_pass"] & (bt.index <= h_max)]
    if len(ok):
        chosen = float(ok[0])
        print(f"selected h={chosen} (smallest with every region_local x version cell >= {min_n}, "
              f"h<={h_max})")
    else:
        chosen = fallback
        print(f"!! no h <= {h_max} reaches {min_n} claims in every cell -- the near-boundary band "
              f"is thin at every local width tried. Keeping fallback h={fallback}; read the local "
              f"DiD below as LOW-POWER, not as a clean estimate.")
    display(bt)
    return chosen, bt


def local_cross_version_estimate(v_before: str, split_before: str, v_after: str, split_after: str,
                                  h: float) -> dict:
    """Same mechanism as cross_version_estimate, but region_local (band [tau-h, tau+h]) instead
    of the whole-population region -- a sharp-RDD robustness check, not a replacement (see the
    markdown above: simpson(B') - simpson(A') at one version's own tau is a boundary
    discontinuity in the GROUP-LEVEL concentration functional, analogous to a LATE, not a literal
    one -- same distinction Eq. eq:boundary-gap already carries)."""
    tb = local_region(version_pair_table(v_before, split_before), v_before, h)
    ta = local_region(version_pair_table(v_after, split_after), v_after, h)

    sA_before, sB_before = cell_simpson_local(tb, region_local="A"), cell_simpson_local(tb, region_local="B")
    sA_after, sB_after = cell_simpson_local(ta, region_local="A"), cell_simpson_local(ta, region_local="B")
    did_local = (sB_after - sA_after) - (sB_before - sA_before)

    n_before = {r: int((tb["region_local"] == r).sum()) for r in ("A", "B")}
    n_after = {r: int((ta["region_local"] == r).sum()) for r in ("A", "B")}
    print(f"{v_before}: simpson(A')={sA_before:.4f}  simpson(B')={sB_before:.4f}  "
          f"n=(A'={n_before['A']:,}, B'={n_before['B']:,})")
    print(f"{v_after}: simpson(A')={sA_after:.4f}  simpson(B')={sB_after:.4f}  "
          f"n=(A'={n_after['A']:,}, B'={n_after['B']:,})")
    print(f"\nlocal DiD({v_before}→{v_after}, h={h}) = {did_local:+.4f}")

    return {"pair": f"{v_before}({split_before})->{v_after}({split_after})", "h": h,
            "did_local": did_local, "n_before": sum(n_before.values()), "n_after": sum(n_after.values())}


chosen_h, local_h_table = select_local_h("v2", OOT["v2"], "v3", OOT["v3"])
figstyle.save_table(local_h_table, "04_02_local_band_h_selection")
print()
local_estimate_v2_v3 = local_cross_version_estimate("v2", OOT["v2"], "v3", OOT["v3"], chosen_h)

local_cross_version_table = pd.DataFrame([local_estimate_v2_v3]).set_index("pair")
figstyle.save_table(local_cross_version_table, "04_02_local_cross_version_estimate")
display(local_cross_version_table)

print(f"\nwhole-population DiD(v2→v3) = {estimate_v2_v3['did']:+.4f}   "
      f"local DiD(v2→v3, h={chosen_h}) = {local_estimate_v2_v3['did_local']:+.4f}")
print("-> same sign and comparable size: the whole-population estimate is not just an artefact of")
print("   region A/B having different covariate distributions.")
print("-> local DiD much smaller or flips sign: the whole-population estimate may be leaning on")
print("   that covariate-distribution difference more than on the forced-label mechanism itself.")

### Per-feature view of the local band cells

Same reasoning as the within-version confound-check figure below: the Simpson numbers above are
one scalar per cell, not which features carry the near-boundary mass. Two figures, one per
version, each comparing region A' (just below τ) against B' (just above τ) — real name and alias,
same convention as everywhere else in this notebook family. Read alongside `00_SHAP.ipynb`
§9c's WITHIN-version near-τ bar chart for the same versions: this is its cross-version companion.

In [ ]:
# Per-feature view of the local band cells (region_local A'/B', chosen h, one version at a time --
# v2 and v3 do NOT share a feature space, so this is deliberately two separate figures/tables,
# never one combined frame indexed by feature (see the wide-table note below).
LOCAL_TOPK_FEATURES = 10
_LOCAL_CELL_COLOUR = {"A": figstyle.SERIES[0], "B": figstyle.SERIES[1]}


def _local_band_features_fig(mabs_by_region: dict, version: str, h: float, aliased: bool) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.6), squeeze=False)
    for ax, region in zip(axes[0], ("A", "B")):
        top = mabs_by_region[region].sort_values(ascending=False).head(LOCAL_TOPK_FEATURES)[::-1]
        labels = feature_alias.to_alias(version, top.index) if aliased else list(top.index)
        bars = ax.barh(np.arange(len(top)), top.values, color=_LOCAL_CELL_COLOUR[region])
        ax.set_yticks(np.arange(len(top)))
        ax.set_yticklabels(labels, fontsize=7)
        ax.set_title(f"region {region}' ({'just below' if region == 'A' else 'just above'} τ)")
        ax.set_xlabel("mean |phi|")
        ax.bar_label(bars, fmt="%.3f", fontsize=6, padding=2)
        ax.margins(x=0.12)     # headroom so the value label never clips past the axes edge
    fig.suptitle(f"{version} · local band (h={h}) — feature attribution just below/above τ"
                 + (" (aliased)" if aliased else ""))
    fig.tight_layout()
    prefix = "alias_" if aliased else ""
    figstyle.save(fig, f"{prefix}{version}_04_02_local_band_cell_features")
    plt.show()


for _v, _split in (("v2", OOT["v2"]), ("v3", OOT["v3"])):
    _band = local_region(version_pair_table(_v, _split), _v, chosen_h)
    _mabs_by_region = {r: cell_mabs_local(_band, region_local=r) for r in ("A", "B")}

    _local_band_features_fig(_mabs_by_region, _v, chosen_h, aliased=False)
    _local_band_features_fig(_mabs_by_region, _v, chosen_h, aliased=True)

    # Full per-feature table (both regions, this version's OWN feature names only -- v2 and v3
    # are never combined into one indexed frame here).
    _wide = pd.DataFrame({r: _mabs_by_region[r] for r in ("A", "B")}).sort_values("B", ascending=False)
    _wide.index.name = "feature"
    figstyle.save_table(_wide, f"{_v}_04_02_local_band_cell_features")
    _alias_wide = _wide.set_axis(feature_alias.to_alias(_v, list(_wide.index)), axis=0).rename_axis("feature")
    figstyle.save_table(_alias_wide, f"alias_{_v}_04_02_local_band_cell_features")

## `within_version_confound_check` — is there drift inside v3's OWN training window?

**Why v3 train, not v2 train, and not v3's OOT.** Two rules apply here, different from the
section above on purpose:

- **This check must run on the TRAIN split**, because train is what actually shapes the model's
  fitted function — if something confounds the DATA used to fit the model, it lives here, not in
  the holdout.
- **v2's train is excluded, not just weaker.** v2's train/val/test (2018-2020) predates v2's own
  deployment (2021-06-03) entirely — decisions on those claims were made under **v1's** rule
  (segmented on mobility), and that population was never fed into v3's training regardless. Worse,
  even a "pretend there's no band, just a hard 0.85 cutoff" version of this check is not
  computable for v2: it would need v1's SCORE on v2's training claims, and that cannot be
  recovered (v1's serving log is destroyed, and v2's own training matrix does not carry v1's
  score). **v3's train is the only version this check can run on at all.**
- **v3's OOT is not used here either** — this section is asking "did the FITTING data carry a
  drift", which is a property of `train` specifically, not a generalisation question (that is
  what `cross_version_estimate` above already covers on OOT).

**Two sub-checks, not one**, because a single median split cannot tell a real regime effect apart
from an unrelated confound (economic conditions, seasonality, channel mix):

1. **Quick check** — median-split the WHOLE v3 train window (crosses the real regime break at
   2024-06-02). A non-zero DiD here could be the regime break OR an unrelated confound OR both.
2. **Within-one-regime check** — median-split ONLY the pre-break stretch (2023-06-02 to
   2024-06-02), where the decision rule did NOT change at all. If DiD is ~0 here but large in (1),
   that points at the regime break specifically. If DiD is ALSO large here, that is a confound
   unrelated to the regime break, and it will bias any cross-version reading too.

**`04_04_regime_break_robustness.ipynb` is the deep dive** once either sub-check flags something —
it re-tags v3 with the REAL break dates (both 2024-06-02 and 2026-02-25, the second one only
inside v3's OOT) instead of a median, and checks whether the conclusion is robust to picking the
real date over an arbitrary one.

In [ ]:
# §3 — v3 train window bounds and the real regime break inside it (derived, never hand-copied --
# see 04_04 §1 for the identical pattern)
def window_bounds(version: str, split: str) -> tuple:
    d = load(version, split=split)
    dates = pd.to_datetime(d.frame[schema.DATE])
    return dates.min(), dates.max() + pd.Timedelta(days=1)


def region_and_era_at(version: str, split: str, cutoff, date_filter=None) -> pd.DataFrame:
    """region(A/B) + an early/late tag at a CHOSEN cutoff -- optionally restricted to a date
    sub-range first (date_filter = (start, end), half-open), for the within-one-regime sub-check.
    """
    d = load(version, split=split)
    df = d.frame.copy()
    df[schema.DECISION] = d.decisions
    df["region"] = np.where(df[schema.DECISION] == 1, "B", "A")
    dates = pd.to_datetime(df[schema.DATE])
    if date_filter is not None:
        start, end = date_filter
        keep = (dates >= pd.Timestamp(start)) & (dates < pd.Timestamp(end))
        df, dates = df[keep], dates[keep]
    cutoff = pd.Timestamp(cutoff)
    df["era"] = np.where(dates <= cutoff, "early", "late")
    return df[[ID_COL, "region", "era"]]


def confound_did(version: str, split: str, cutoff, date_filter=None) -> dict:
    tags = region_and_era_at(version, split, cutoff, date_filter)
    attrs = load(version, split=split).attributions
    table = tags.merge(attrs, on=ID_COL, how="inner")
    s = {(r, e): cell_simpson(table, region=r, era=e) for r in ("A", "B") for e in ("early", "late")}
    did = (s[("B", "late")] - s[("A", "late")]) - (s[("B", "early")] - s[("A", "early")])
    return {"cutoff": pd.Timestamp(cutoff).date(), "did": did, "n_claims": len(table),
            "n_early": int((tags["era"] == "early").sum()), "n_late": int((tags["era"] == "late").sum())}


train_start, train_end = window_bounds("v3", "train")
v3_breaks = [b["date"] for b in config.spans_a_break("v2", train_start, train_end)]
print(f"v3 train window: {train_start.date()} .. {train_end.date()}")
print(f"v2 regime break(s) inside it: {v3_breaks}")
assert len(v3_breaks) == 1, f"expected exactly one break in v3 train, found {v3_breaks} -- update this cell"
break_date = v3_breaks[0]

In [ ]:
# §3a — quick check: median split of the WHOLE v3 train window
whole_median = pd.Timestamp(train_start) + (pd.Timestamp(train_end) - pd.Timestamp(train_start)) / 2
quick_check = confound_did("v3", "train", whole_median)
quick_check["kind"] = "quick (whole train, median cutoff)"
print("quick check (whole v3 train, median split):")
print(quick_check)

# §3b — within-one-regime check: median split of ONLY the pre-break stretch, where the decision
# rule did not change at all -- expected DiD ~= 0 if there is no confound
pre_break_median = pd.Timestamp(train_start) + (pd.Timestamp(break_date) - pd.Timestamp(train_start)) / 2
within_regime_check = confound_did("v3", "train", pre_break_median,
                                    date_filter=(train_start, break_date))
within_regime_check["kind"] = f"within one regime (pre-{break_date}, median cutoff)"
print("\nwithin-one-regime check (pre-break stretch only, median split):")
print(within_regime_check)

confound_table = pd.DataFrame([quick_check, within_regime_check]).set_index("kind")
figstyle.save_table(confound_table, "04_02_within_version_confound_check")
display(confound_table)

print("\ninterpretation:")
print("  quick check large + within-regime ~0  -> points at the regime break itself (see 04_04)")
print("  both large                            -> a confound unrelated to the regime break too")
print("  both ~0                               -> no evidence of drift in v3's own training window")

### Per-feature view of the within-version confound-check cells

Same reasoning as 04_01 §6 / the old §2b: the Simpson numbers above are one scalar per cell, not
which features moved. This plots mean|phi| per feature in each of the 4 (region x era) cells of
the **quick check** (whole v3 train, median split) — real name and alias, two independent figures
and two CSVs, same convention as everywhere else in this notebook family.

In [ ]:
# Per-feature view for the quick-check cells (region x era, whole v3 train, median cutoff)
CELLS = (("A", "early"), ("A", "late"), ("B", "early"), ("B", "late"))
TOPK_FEATURES = 10
_CELL_COLOUR = {"A": figstyle.SERIES[0], "B": figstyle.SERIES[1]}


def _cell_features_fig(mabs_by_cell: dict, version: str, label: str, aliased: bool) -> None:
    fig, axes = plt.subplots(1, len(CELLS), figsize=(4.2 * len(CELLS), 3.6), squeeze=False)
    for ax, (region, era) in zip(axes[0], CELLS):
        top = mabs_by_cell[(region, era)].sort_values(ascending=False).head(TOPK_FEATURES)[::-1]
        labels = feature_alias.to_alias(version, top.index) if aliased else list(top.index)
        bars = ax.barh(np.arange(len(top)), top.values, color=_CELL_COLOUR[region])
        ax.set_yticks(np.arange(len(top)))
        ax.set_yticklabels(labels, fontsize=7)
        ax.set_title(f"region {region} · {era}")
        ax.set_xlabel("mean |phi|")
        ax.bar_label(bars, fmt="%.3f", fontsize=6, padding=2)
        ax.margins(x=0.12)     # headroom so the value label never clips past the axes edge
    fig.suptitle(f"{version} · {label} — feature attribution by cell"
                 + (" (aliased)" if aliased else ""))
    fig.tight_layout()
    prefix = "alias_" if aliased else ""
    figstyle.save(fig, f"{prefix}v3_train_04_02_confound_check_cell_features")
    plt.show()


tags_quick = region_and_era_at("v3", "train", whole_median)
attrs_v3_train = load("v3", split="train").attributions
table_quick = tags_quick.merge(attrs_v3_train, on=ID_COL, how="inner")
mabs_by_cell = {cell: cell_mabs(table_quick, region=cell[0], era=cell[1]) for cell in CELLS}

_cell_features_fig(mabs_by_cell, "v3", "train (quick check)", aliased=False)
_cell_features_fig(mabs_by_cell, "v3", "train (quick check)", aliased=True)

wide = pd.DataFrame({f"{r}_{e}": mabs_by_cell[(r, e)] for r, e in CELLS}).sort_values("B_late", ascending=False)
wide.index.name = "feature"
figstyle.save_table(wide, "v3_train_04_02_confound_check_cell_features")
alias_wide = wide.set_axis(feature_alias.to_alias("v3", list(wide.index)), axis=0).rename_axis("feature")
figstyle.save_table(alias_wide, "alias_v3_train_04_02_confound_check_cell_features")

## v2↔v3 shared claims — score drift + a combined dose table

`corrector_targets` for v3 already joins v3's own claims to **v2's recorded score AND decision**
(from v2's serving log, `log_scores`) by claim_id — this is not a new join, `03_01_corrector_inputs.ipynb`
already built it. So for every v3 claim that has a corrector_targets row, we have BOTH v2's score
(what the old model would have said) and v3's own score (what the new model actually says) on the
exact same claim — a genuinely paired comparison, no case-mix confound, for the SCORE level (not
yet SHAP — that needs `log_features`, which is the stretch goal below).

In [ ]:
# §4 — score drift: v2's score (from corrector_targets, i.e. v2's log) vs v3's own score, on the
# SAME v3 claims, for every v3 split that has a corrector_targets file
drift_rows = []
for split in ("train", config.OOT_SPLIT["v3"]):
    ct_path = config.split_path("corrector_targets", "v3", split)
    if not ct_path.is_file():
        print(f"[v3/{split}] no corrector_targets -- skip")
        continue
    ct = pd.read_parquet(ct_path)[[ID_COL, "score", schema.DECISION]].rename(
        columns={"score": "v2_score", schema.DECISION: "v2_decision"})
    v3_own = load("v3", split=split).frame[[ID_COL, "model_v3_score"]].rename(
        columns={"model_v3_score": "v3_score"})
    merged = ct.merge(v3_own, on=ID_COL, how="inner")

    corr = merged["v2_score"].corr(merged["v3_score"])
    mean_diff = (merged["v3_score"] - merged["v2_score"]).mean()

    drift_rows.append({"split": split, "n_shared": len(merged), "score_corr": corr,
                        "mean_v3_minus_v2": mean_diff})
    print(f"[v3/{split}] n_shared={len(merged):,}  corr(v2_score, v3_score)={corr:.4f}  "
          f"mean(v3-v2)={mean_diff:+.4f}")

    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.scatter(merged["v2_score"], merged["v3_score"], s=4, alpha=0.3, color=figstyle.SERIES[0])
    ax.set_xlabel("v2 score (this claim, from v2\'s log)")
    ax.set_ylabel("v3 score (this claim, v3\'s own model)")
    ax.set_title(f"v2 vs v3 score, shared claims \u2014 {split}")
    fig.tight_layout()
    figstyle.save(fig, f"v3_{split}_04_02_score_drift_scatter")
    plt.show()

    figstyle.save_table(merged, f"v3_{split}_04_02_shared_claims_scores")

drift_table = pd.DataFrame(drift_rows).set_index("split")
figstyle.save_table(drift_table.join(dose_table.xs("v3", level="version"), how="left"),
                     "04_02_v2_v3_score_drift_and_dose")
display(drift_table)

### Stretch goal, NOT built (time-permitting only): full SHAP comparison on shared claims

The score-level comparison above needs only `corrector_targets` (already exists). A **SHAP-level**
shared-claims comparison — running v2's OWN model on the exact same v3 claims, to get v2's SHAP on
them — additionally needs `log_features` (v2's serving-time feature matrix, same log grain as
`log_scores`) to actually cover the calendar window these v3 claims fall in, AND a new attribution
run against that log grain (`scoring/attribute_all.py` currently only reads `processed_inputs`,
the train/val/test grain — see `project_v2v3_shared_claims_corrector_targets`). This is real new
data-engineering work on the company laptop, not something this notebook can do blind. Left as a
placeholder section on purpose — revisit only if time allows against the thesis deadline.

## Notes

- **`cross_version_estimate`**: one mechanism, region(A/B) x version(before/after), each version
  on its OWN OOT split. v1->v2 and v2->v3 use the identical function; which role a given call's
  DiD plays (validity check vs estimate) is read from the printed dose numbers by hand, not
  auto-classified (an earlier auto-threshold attempt broke on v1's dose being exactly 0).
- **`local_cross_version_estimate` (2026-09-13) is a sharp Regression Discontinuity Design (RDD),
  analogous to a Local Average Treatment Effect (LATE), not a literal one.** Restricting to
  `region_local` A'/B' (`|score - tau| <= h`) is the bandwidth restriction a sharp RDD uses, and
  `simpson(B') - simpson(A')` for one version at its own τ is a boundary discontinuity in that
  version's own τ — exactly the construction the thesis already commits to for
  $\Delta\phi_{\mathrm{boundary}}$ (`eq:boundary-gap`, `sec:bg-causal`): "a SHAP-space analogue of
  $\tau_{\mathrm{SRD}}$ itself ... in place of an outcome". It is an analogue of a LATE, not a
  literal one, because $C_B-C_A$ is built from each side's POOLED mean-$|\phi|$ profile
  (`eq:meanabsshap`, the thesis's own group-level headline definition — used identically in the
  whole-population numbers above), never from an average of each claim's OWN concentration:
  Simpson is convex, so "Simpson of the group average" and "average of each claim's own Simpson"
  can move in different directions (e.g. two claims each 100% on a different single feature read
  as maximally concentrated individually, but pooled first their average profile splits 50/50 and
  reads as LESS concentrated). Kept group-level on purpose, so it stays comparable to
  `cross_version_estimate`'s whole-population number, which uses the identical definition.
  `local_cross_version_estimate` DiDs two such boundary discontinuities across v2->v3. `h` is
  grid-selected on cell counts alone (never on a concentration number), same
  power-gate/locality-cap discipline as `03_02` SS2c's `band_h` selection, re-derived here rather
  than importing that notebook's number. Read this next to the whole-population estimate as a
  check on whether A/B's differing covariate distributions, not the forced-label mechanism, are
  doing the work (`subsec:shap-did`'s parallel-trends caveat) — it does not replace
  `cross_version_estimate`, it bounds how much of it survives once the two sides are forced to
  look almost identical except for which side of τ they fell on.
- **`within_version_confound_check`**: v3 TRAIN only (never v2 train — wrong population, and not
  even computable there; never v3's OOT — this checks the FITTING data specifically). Two
  sub-checks: quick (whole window, median) and within-one-regime (pre-break stretch only, median)
  — the gap between them is what separates "the regime break did it" from "an unrelated confound
  did it". `04_04_regime_break_robustness.ipynb` is the deep dive once this flags something.
- **v2↔v3 shared claims (score level)**: uses `corrector_targets` as-is, no new join. The
  SHAP-level version is a stretch goal, not built.
- **Reproducibility audit (2026-09-13).** Every computation in this notebook is deterministic
  (grid searches over cell COUNTS, joins, Simpson/mean|phi| arithmetic) — nothing here samples or
  shuffles, so nothing here needs a seed. The one random component upstream of this notebook is
  the SHAP attribution itself (`00_SHAP.ipynb`'s background/explain-row sampling), which already
  sets `SEED = 0` via `np.random.RandomState(SEED)` and reuses it for `interaction_values` too —
  confirmed present, not something this notebook needs to touch.
- **Every table above is saved via `figstyle.save_table()`** (CSV, `figures/`) and every figure via
  `figstyle.save()` (PNG) — real name AND alias twin wherever a feature name appears on an axis.
  Nothing here is display()-only.
- **Feature names are never printed raw without an alias twin** — route any future figure/table
  through `feature_alias.to_alias()` the same way the two per-feature sections above do.
- Not yet run against real data (none available locally). Cell counts, thin-partition warnings,
  and every number above are unverified until run on the company laptop.